# Domestic and Family Violence in Australia

Working hypothesis: police-recorded incidence of domestic and family violence has risen substantially over the past decade or two, but the *underlying* rate has changed little. The plan is to triangulate from harder outcomes (deaths, hospitalisations) against the softer measure of police-recorded offences.

Sections:
1. **Deaths** —
   - 1A. ABS *Causes of Death* (3303.0): all assault deaths, 2015-2024
   - 1B. AIC *National Homicide Monitoring Program* via AIHW: domestic homicide victims, 1989-90 to 2024-25
2. **Hospitalisations** — AIHW *National Hospital Morbidity Database*: FDV hospitalisations, 2019-20 to 2023-24
3. **Police-recorded** — ABS *Recorded Crime - Victims* (4510.0): family/domestic violence flag (TODO)
4. **Combined indexed view** putting (1)-(3) on one chart (TODO)

## Python set-up

In [1]:
# stdlib
import urllib.request
from pathlib import Path

# analytic imports
import numpy as np
import pandas as pd
import readabs as ra
import mgplot as mg

# local imports
from abs_helper import get_population

# pandas display settings
pd.options.display.max_rows = 9999
pd.options.display.max_columns = 999
pd.options.display.max_colwidth = 100

# notebook constants
SHOW = False
SOURCE_DEATHS = "ABS 3303.0"
SOURCE_NHMP = "AIC NHMP via AIHW"
SOURCE_HOSP = "AIHW NHMD"
SOURCE_POLICE = "ABS 4510.0"
LFOOTER = "Australia. "
FDV_NOTE = "FDV = Family/Domestic Violence. "

In [2]:
# chart directory
CHART_DIR = "./CHARTS/Domestic and Family Violence/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()

In [3]:
# helpers for the ABS Causes of Death (3303.0) data cubes

def find_table(tables: dict, workbook_substr: str, sheet: str) -> pd.DataFrame:
    """Look up a sheet inside a 3303.0 workbook, ignoring release year.
    Workbook keys look like '2024_01 Underlying causes of death (Australia)---Table 1.2'."""
    suffix = f"---{sheet}"
    candidates = [k for k in tables if workbook_substr in k and k.endswith(suffix)]
    if not candidates:
        raise KeyError(f"No sheet matching {workbook_substr!r} + {sheet!r}")
    if len(candidates) > 1:
        raise KeyError(f"Ambiguous: {candidates}")
    return tables[candidates[0]]


def extract_year_df(df: pd.DataFrame, row_label: str) -> pd.DataFrame:
    """Extract a Males/Females/Persons-by-year DataFrame from the Causes of Death
    Table 1.2 layout. Row 3 holds merged year headers, row 4 holds the sex columns.
    Index is annual PeriodIndex (mgplot requires PeriodIndex/RangeIndex)."""
    years = pd.Series(df.iloc[3].tolist()).ffill()
    sexes = df.iloc[4].tolist()
    matches = df.iloc[:, 0].astype(str).str.strip() == row_label
    if not matches.any():
        raise KeyError(f"Row not found: {row_label!r}")
    row = df[matches].iloc[0]
    out = pd.DataFrame(
        {
            sex: {
                int(years.iloc[c]): row.iloc[c]
                for c, s in enumerate(sexes) if s == sex
            }
            for sex in ("Males", "Females", "Persons")
        }
    ).astype(float).sort_index()
    out.index = pd.PeriodIndex(out.index.astype(int), freq="Y")
    return out

In [4]:
# helpers for the AIHW Family, Domestic and Sexual Violence workbook

AIHW_FDSV_URL = (
    "https://www.aihw.gov.au/getmedia/"
    "f4a9196a-9797-4b03-acc8-d72a5b83d370/"
    "AIHW-FDSV-all-data-download.xlsx"
)
AIHW_FDSV_CACHE = Path("./CACHE/AIHW_FDSV/all-data-download.xlsx")


def fetch_aihw_fdsv(refresh: bool = False) -> Path:
    """Download the AIHW FDSV all-data workbook to the local cache and return its path."""
    if refresh or not AIHW_FDSV_CACHE.exists():
        AIHW_FDSV_CACHE.parent.mkdir(parents=True, exist_ok=True)
        req = urllib.request.Request(
            AIHW_FDSV_URL, headers={"User-Agent": "Mozilla/5.0"}
        )
        with urllib.request.urlopen(req, timeout=60) as r:
            AIHW_FDSV_CACHE.write_bytes(r.read())
    return AIHW_FDSV_CACHE


def fy_end_year(fy: str) -> int:
    """Convert AIHW financial year string (e.g. '2019\u201320', '1999\u20132000') to its end year."""
    return int(fy.split("\u2013")[0]) + 1


def fit_trend(s: pd.Series) -> pd.Series:
    """OLS linear trend across a PeriodIndex series; returns predicted values."""
    s = s.dropna()
    x = s.index.year.to_numpy(dtype=float)
    slope, intercept = np.polyfit(x, s.to_numpy(dtype=float), 1)
    return pd.Series(slope * x + intercept, index=s.index)

## 1A. All assault deaths from ABS Causes of Death (3303.0)

ABS publishes assault deaths (ICD-10 X85-Y09) annually in the *Underlying causes of death (Australia)* data cube, Table 1.2. The published time series in the latest release covers the most recent 10 years.

Two important caveats:
- **Source is death registrations, not hospital data.** The ABS codes the underlying cause from death certificates; the death may have occurred anywhere.
- **All assault deaths, not only FDV.** AIC's National Homicide Monitoring Program (Section 1B below) is the better FDV-specific source — these ABS figures are included for the broader assault context.

In [5]:
# fetch all Causes of Death workbooks (slow first time, then cached)
cod_url = (
    "https://www.abs.gov.au/statistics/health/causes-death/"
    "causes-death-australia/latest-release"
)
cod_tables = ra.grab_abs_url(url=cod_url)
len(cod_tables)

171

In [6]:
# extract assault deaths (X85-Y09) by year and sex
underlying_aus = find_table(
    cod_tables,
    "Underlying causes of death (Australia)",
    "Table 1.2",
)
assault_df = extract_year_df(underlying_aus, "Assault (X85-Y09)")
assault_df

,Males,Females,Persons
2015,189.0,94.0,283.0
2016,180.0,85.0,265.0
2017,145.0,64.0,209.0
2018,169.0,83.0,252.0
2019,186.0,78.0,264.0
2020,169.0,81.0,250.0
2021,159.0,68.0,227.0
2022,163.0,73.0,236.0
2023,164.0,76.0,240.0
2024,167.0,100.0,267.0


In [7]:
# bar chart: assault deaths per year (total persons)
mg.bar_plot_finalise(
    assault_df["Persons"],
    title="Assault Deaths: Australia",
    ylabel="Number of deaths per year",
    rfooter=SOURCE_DEATHS,
    lfooter=LFOOTER + "ICD-10 X85-Y09. Underlying cause of death.",
    annotate=True,
    rounding=0,
    label_rotation=0,
    show=SHOW,
)

In [8]:
# line chart: assault deaths by sex
mg.line_plot_finalise(
    assault_df[["Males", "Females"]],
    title="Assault Deaths by Sex: Australia",
    ylabel="Number of deaths per year",
    rfooter=SOURCE_DEATHS,
    lfooter=LFOOTER + "ICD-10 X85-Y09. Underlying cause of death.",
    width=2,
    color=["cornflowerblue", "hotpink"],
    marker="o",
    annotate=True,
    rounding=0,
    show=SHOW,
)

In [9]:
# compute rate per 100,000 using mid-year (June quarter) ERP
pop_q, _ = get_population(state="Australia", project=False)
mid_year_pop = pop_q[pop_q.index.quarter == 2].copy()
mid_year_pop.index = pd.PeriodIndex(mid_year_pop.index.year, freq="Y")

assault_rate = (assault_df["Persons"] / mid_year_pop * 100_000).dropna()
assault_rate.name = "Assault deaths per 100,000"
assault_rate

2015    1.188277
2016    1.095453
2017    0.849850
2018    1.009484
2019    1.042044
2020    0.974687
2021    0.883770
2022    0.907039
2023    0.900228
2024    0.981824
Freq: Y-DEC, Name: Assault deaths per 100,000, dtype: float64

In [10]:
# line chart: assault death rate per 100,000
mg.line_plot_finalise(
    assault_rate,
    title="Assault Death Rate: Australia",
    ylabel="Deaths per 100,000 persons",
    rfooter=f"{SOURCE_DEATHS}, ABS 3101.0",
    lfooter=LFOOTER + "ICD-10 X85-Y09. Mid-year ERP denominator.",
    width=2,
    color=["darkred"],
    marker="o",
    annotate=True,
    rounding=2,
    show=SHOW,
)

## 1B. Domestic homicide victims (AIC National Homicide Monitoring Program)

Source: Australian Institute of Criminology *National Homicide Monitoring Program* (NHMP), surfaced through AIHW sheet NHMP 2. **35 years of FDV-specific homicide data** (1989-90 to 2024-25) — far longer than the 10-year ABS *Causes of Death* window above, and scoped specifically to domestic homicides (intimate partner, family and other domestic relationships).

Notes from AIHW:
- A `*` flag in the source data indicates a calculated rate based on fewer than 20 events in the numerator and may be unstable.
- Persons totals include victims where sex was not stated or unknown; Males + Females will not exactly equal Persons.
- Year ending 30 June (financial year).
- Intimate partner homicide rates use the **18+ population** as the denominator (per AIHW methodology note 6); rates are sex-specific.

In [11]:
# fetch AIHW workbook (cached) and parse NHMP 2 (Domestic homicide victims)
aihw_xlsx = fetch_aihw_fdsv()
nhmp2 = pd.read_excel(aihw_xlsx, sheet_name="NHMP 2", header=8)
nhmp2["YearEnd"] = nhmp2["Period"].apply(fy_end_year)


def pivot_nhmp(homicide_type: str, unit: str) -> pd.DataFrame:
    sub = nhmp2[
        (nhmp2["Type of homicide"] == homicide_type)
        & (nhmp2["Unit"] == unit)
    ]
    out = (
        sub.pivot(index="YearEnd", columns="Sex", values="Value")
        .astype(float)
        .sort_index()
    )
    out.index = pd.PeriodIndex(out.index.astype(int), freq="Y")
    return out


dom_count = pivot_nhmp("Domestic", "Number")
dom_rate = pivot_nhmp("Domestic", "Rate (Number per 100,000)")
ipv_count = pivot_nhmp("Intimate partner", "Number")
ipv_rate = pivot_nhmp("Intimate partner", "Rate (Number per 100,000)")
dom_count.tail()

Sex,Females,Males,Persons
YearEnd,,,
2021,48.0,33.0,81.0
2022,42.0,24.0,66.0
2023,48.0,39.0,87.0
2024,60.0,34.0,94.0
2025,48.0,49.0,98.0


In [12]:
# line chart: domestic homicide victims by sex (35 yrs) + linear trend
dom_count_main = dom_count[["Males", "Females"]]
dom_count_trend = pd.DataFrame({
    "Males (trend)": fit_trend(dom_count["Males"]),
    "Females (trend)": fit_trend(dom_count["Females"]),
})

ax = mg.line_plot(
    dom_count_main,
    color=["cornflowerblue", "hotpink"],
    width=2,
    annotate=True,
    rounding=0,
)
mg.line_plot(
    dom_count_trend,
    ax=ax,
    color=["cornflowerblue", "hotpink"],
    width=1,
    style="--",
    annotate=False,
)
mg.finalise_plot(
    ax,
    title="Domestic Homicide Victims by Sex: Australia",
    ylabel="Victims per year",
    rfooter=SOURCE_NHMP,
    lfooter=LFOOTER + "Year ending 30 June. Includes intimate partner, family and other domestic homicides. Dashed lines = OLS linear trend.",
    legend={"loc": "upper right", "fontsize": "small"},
    show=SHOW,
)

In [13]:
# line chart: domestic homicide RATE by sex (35 yrs) + linear trend
dom_rate_main = dom_rate[["Males", "Females"]]
dom_rate_trend = pd.DataFrame({
    "Males (trend)": fit_trend(dom_rate["Males"]),
    "Females (trend)": fit_trend(dom_rate["Females"]),
})

ax = mg.line_plot(
    dom_rate_main,
    color=["cornflowerblue", "hotpink"],
    width=2,
    annotate=True,
    rounding=2,
)
mg.line_plot(
    dom_rate_trend,
    ax=ax,
    color=["cornflowerblue", "hotpink"],
    width=1,
    style="--",
    annotate=False,
)
mg.finalise_plot(
    ax,
    title="Domestic Homicide Rate by Sex: Australia",
    ylabel="Victims per 100,000\nsame-sex population",
    rfooter=SOURCE_NHMP,
    lfooter=LFOOTER + "Year ending 30 June. Sex-specific crude rate. Dashed lines = OLS linear trend.",
    legend={"loc": "upper right", "fontsize": "small"},
    show=SHOW,
)

In [14]:
# line chart: intimate partner homicide victims by sex (35 yrs) + linear trend
ipv_count_main = ipv_count[["Males", "Females"]]
ipv_count_trend = pd.DataFrame({
    "Males (trend)": fit_trend(ipv_count["Males"]),
    "Females (trend)": fit_trend(ipv_count["Females"]),
})

ax = mg.line_plot(
    ipv_count_main,
    color=["cornflowerblue", "hotpink"],
    width=2,
    annotate=True,
    rounding=0,
)
mg.line_plot(
    ipv_count_trend,
    ax=ax,
    color=["cornflowerblue", "hotpink"],
    width=1,
    style="--",
    annotate=False,
)
mg.finalise_plot(
    ax,
    title="Intimate Partner Homicide Victims by Sex: Australia",
    ylabel="Victims per year",
    rfooter=SOURCE_NHMP,
    lfooter=LFOOTER + "Year ending 30 June. Intimate partner homicides only (subset of all domestic). Dashed lines = OLS linear trend.",
    legend={"loc": "upper right", "fontsize": "small"},
    show=SHOW,
)

In [15]:
# line chart: intimate partner homicide RATE by sex (35 yrs) + linear trend
ipv_rate_main = ipv_rate[["Males", "Females"]]
ipv_rate_trend = pd.DataFrame({
    "Males (trend)": fit_trend(ipv_rate["Males"]),
    "Females (trend)": fit_trend(ipv_rate["Females"]),
})

ax = mg.line_plot(
    ipv_rate_main,
    color=["cornflowerblue", "hotpink"],
    width=2,
    annotate=True,
    rounding=2,
)
mg.line_plot(
    ipv_rate_trend,
    ax=ax,
    color=["cornflowerblue", "hotpink"],
    width=1,
    style="--",
    annotate=False,
)
mg.finalise_plot(
    ax,
    title="Intimate Partner Homicide Rate by Sex: Australia",
    ylabel="Victims per 100,000\nsame-sex population aged 18+",
    rfooter=SOURCE_NHMP,
    lfooter=LFOOTER + "Year ending 30 June. Sex-specific crude rate. Dashed lines = OLS linear trend.",
    legend={"loc": "upper right", "fontsize": "small"},
    show=SHOW,
)

In [16]:
# line chart: domestic homicide rate per 100,000 (Persons)
mg.line_plot_finalise(
    dom_rate["Persons"],
    title="Domestic Homicide Rate: Australia",
    ylabel="Victims per 100,000 population",
    rfooter=SOURCE_NHMP,
    lfooter=LFOOTER + "Year ending 30 June. Persons total (includes unknown sex).",
    width=2,
    color=["darkred"],
    annotate=True,
    rounding=2,
    show=SHOW,
)

In [17]:
# line chart: domestic vs intimate partner homicide rates (Persons)
compare = pd.DataFrame({
    "All domestic": dom_rate["Persons"],
    "Intimate partner": ipv_rate["Persons"],
})
mg.line_plot_finalise(
    compare,
    title="Domestic vs Intimate Partner Homicide Rate: Australia",
    ylabel="Victims per 100,000 population",
    rfooter=SOURCE_NHMP,
    lfooter=LFOOTER + "Year ending 30 June. Persons total. IPV is a subset of all domestic.",
    width=2,
    color=["darkred", "darkorange"],
    annotate=True,
    rounding=2,
    show=SHOW,
)

## 2. Hospitalisations from AIHW (NHMD)

Source: AIHW *National Hospital Morbidity Database*, surfaced through the AIHW *Family, domestic and sexual violence* report (sheet NHMD 1). Annual hospital separations where the principal diagnosis was assault and the perpetrator was recorded as a spouse, domestic partner, parent or other family member.

Caveats from the AIHW notes:
- **Five years only** (2019-20 to 2023-24). Perpetrator coding has only become reliable enough for time series in recent years; the proportion of assault hospitalisations with the perpetrator specified has been rising since coding was introduced in 2002-03, which can inflate apparent trends.
- **NSW break in series at 2017-18** (predates this window).
- **'All FDV perpetrators' is de-duplicated** within sex; counts across perpetrator subcategories are not additive.
- **Financial-year annual** (year ending 30 June) — indexed here to the end year for compatibility with the ABS calendar-year deaths series.

In [18]:
# parse NHMD 1 (FDV hospitalisations) from the cached AIHW workbook
nhmd1 = pd.read_excel(aihw_xlsx, sheet_name="NHMD 1", header=14)

# filter to All ages, All FDV perpetrators
fdv = nhmd1[
    (nhmd1["Age group"] == "All ages")
    & (nhmd1["Perpetrator's relationship to victim"] == "All FDV perpetrators")
].copy()
fdv["Value"] = fdv["Value"].astype(str).str.replace(",", "").astype(float)
fdv["YearEnd"] = fdv["Year"].apply(fy_end_year)


def pivot_unit(unit: str) -> pd.DataFrame:
    out = (
        fdv[fdv["Unit"] == unit]
        .pivot(index="YearEnd", columns="Sex", values="Value")
        .sort_index()
    )
    out.index = pd.PeriodIndex(out.index.astype(int), freq="Y")
    return out


fdv_hosp = pivot_unit("Number")
fdv_rate = pivot_unit("Rate per 100,000 population")
fdv_hosp

Sex,Female,Male,Persons
YearEnd,,,
2020,6941.0,2393.0,9334.0
2021,7069.0,2423.0,9493.0
2022,6030.0,2057.0,8088.0
2023,6341.0,2088.0,8432.0
2024,7032.0,2251.0,9291.0


In [19]:
# line chart: FDV hospitalisations by sex (count)
mg.line_plot_finalise(
    fdv_hosp[["Male", "Female"]],
    title="FDV Hospitalisations by Sex: Australia",
    ylabel="Hospitalisations per year",
    rfooter=SOURCE_HOSP,
    lfooter=LFOOTER + FDV_NOTE + "All ages. Year ending 30 June. Spouse/partner or family perpetrator.",
    width=2,
    color=["cornflowerblue", "hotpink"],
    marker="o",
    annotate=True,
    rounding=0,
    show=SHOW,
)

In [20]:
# line chart: FDV hospitalisation rate per 100,000 by sex
mg.line_plot_finalise(
    fdv_rate[["Male", "Female"]],
    title="FDV Hospitalisation Rate by Sex: Australia",
    ylabel="Hospitalisations per\n100,000 same-sex population",
    rfooter=SOURCE_HOSP,
    lfooter=LFOOTER + FDV_NOTE + "All ages. Year ending 30 June. Crude rate.",
    width=2,
    color=["cornflowerblue", "hotpink"],
    marker="o",
    annotate=True,
    rounding=1,
    show=SHOW,
)

## 3. Police-recorded offences from ABS Recorded Crime - Victims (4510.0)

Source: ABS *Recorded Crime - Victims* (4510.0), surfaced through AIHW sheet RCV 1 — FDV-flagged victims of selected offences, 2014-2024 (11 years), by state and sex.

Caveats from the AIHW notes:
- **FDV assault data are not available for Victoria or Australia** (and not published for Queensland prior to 2023). National time series for assault is therefore unavailable from this source.
- **Sexual assault and homicide are nationally comparable** across the full 11-year window.
- **Police-recorded counts depend on victim reporting and police recording practices.** Both have changed substantially over the period (specialist FDV police units, social and legal shifts in willingness to report) — which is exactly the question we are testing here. The expectation is that police-recorded sexual assault will rise much faster than the harder outcome measures in Sections 1-2.

In [21]:
# parse RCV 1 (FDV-flagged Recorded Crime Victims) from the cached AIHW workbook
rcv1 = pd.read_excel(aihw_xlsx, sheet_name="RCV 1", header=12)


def pivot_rcv1(offence: str, unit: str, location: str = "Australia") -> pd.DataFrame:
    """Year x Sex pivot of RCV 1 for a specific offence / unit / location.
    Coerces 'n.a.' / 'n.p.' to NaN and strips comma thousands separators."""
    sub = rcv1[
        (rcv1["Location"] == location)
        & (rcv1["Offence"] == offence)
        & (rcv1["Unit"] == unit)
    ].copy()
    sub["Value"] = pd.to_numeric(
        sub["Value"].astype(str).str.replace(",", ""), errors="coerce"
    )
    out = sub.pivot(index="Year", columns="Sex", values="Value").sort_index()
    out.index = pd.PeriodIndex(out.index.astype(int), freq="Y")
    return out


sa_count = pivot_rcv1("Sexual assault", "Number")
sa_rate = pivot_rcv1("Sexual assault", "Victimisation rate (number per 100,000)")
hom_count = pivot_rcv1("Homicide and related offences", "Number")
sa_count

Sex,Females,Males,Persons
Year,,,
2014,5811,1138,6961
2015,6497,1190,7719
2016,6938,1243,8197
2017,7456,1212,8688
2018,7645,1156,8826
2019,7690,1253,8985
2020,8739,1413,10175
2021,10063,1280,11362
2022,10633,1288,11944


In [22]:
# line chart: FDV sexual assault victims by sex (police data) + linear trend
sa_main = sa_count[["Males", "Females"]]
sa_trend = pd.DataFrame({
    "Males (trend)": fit_trend(sa_count["Males"]),
    "Females (trend)": fit_trend(sa_count["Females"]),
})

ax = mg.line_plot(
    sa_main,
    color=["cornflowerblue", "hotpink"],
    width=2,
    annotate=True,
    rounding=0,
)
mg.line_plot(
    sa_trend,
    ax=ax,
    color=["cornflowerblue", "hotpink"],
    width=1,
    style="--",
    annotate=False,
)
mg.finalise_plot(
    ax,
    title="Police-Recorded FDV Sexual Assault Victims by Sex: Australia",
    ylabel="Police-recorded victims per year",
    rfooter=SOURCE_POLICE,
    lfooter=LFOOTER + FDV_NOTE + "Calendar year. Dashed lines = OLS linear trend.",
    legend={"loc": "upper left", "fontsize": "small"},
    show=SHOW,
)

In [23]:
# line chart: FDV sexual assault victimisation rate by sex (police data) + linear trend
sa_rate_main = sa_rate[["Males", "Females"]]
sa_rate_trend = pd.DataFrame({
    "Males (trend)": fit_trend(sa_rate["Males"]),
    "Females (trend)": fit_trend(sa_rate["Females"]),
})

ax = mg.line_plot(
    sa_rate_main,
    color=["cornflowerblue", "hotpink"],
    width=2,
    annotate=True,
    rounding=1,
)
mg.line_plot(
    sa_rate_trend,
    ax=ax,
    color=["cornflowerblue", "hotpink"],
    width=1,
    style="--",
    annotate=False,
)
mg.finalise_plot(
    ax,
    title="Police-Recorded FDV Sexual Assault Rate by Sex: Australia",
    ylabel="Police-recorded victims per\n100,000 same-sex population",
    rfooter=SOURCE_POLICE,
    lfooter=LFOOTER + FDV_NOTE + "Calendar year. Crude rate. Dashed lines = OLS linear trend.",
    legend={"loc": "upper left", "fontsize": "small"},
    show=SHOW,
)

In [24]:
# parse RCV 6 (total police-recorded sexual assault, all Australia, Persons)
rcv6 = pd.read_excel(aihw_xlsx, sheet_name="RCV 6", header=7)
all_sa_rows = rcv6[
    (rcv6["Characteristic"] == "Sex")
    & (rcv6["Subcategory"] == "Persons")
    & (rcv6["Unit"] == "Number")
].copy()
all_sa_rows["Value"] = pd.to_numeric(
    all_sa_rows["Value"].astype(str).str.replace(",", ""), errors="coerce"
)
all_sa = all_sa_rows.set_index("Year")["Value"].sort_index()
all_sa.index = pd.PeriodIndex(all_sa.index.astype(int), freq="Y")

# overlay FDV-flagged subset (from RCV 1, already parsed above)
compare_sa = pd.DataFrame({
    "All sexual assault": all_sa,
    "FDV-flagged subset": sa_count["Persons"],
})
compare_sa.tail()

,All sexual assault,FDV-flagged subset
Year,,
2020,27538,10175.0
2021,31074,11362.0
2022,32771,11944.0
2023,36352,14069.0
2024,40087,16281.0


In [25]:
# line chart: police-recorded sexual assault — all victims vs FDV-flagged subset
mg.line_plot_finalise(
    compare_sa,
    title="Police-Recorded Sexual Assault: All vs FDV-Flagged",
    ylabel="Victims per year (Persons)",
    rfooter=SOURCE_POLICE,
    lfooter=LFOOTER + FDV_NOTE + "Calendar year. FDV-flagged is a subset of all police-recorded sexual assault.",
    width=2,
    color=["darkred", "darkorange"],
    annotate=True,
    rounding=0,
    show=SHOW,
)

In [26]:
# line chart: FDV-flagged share of all police-recorded sexual assault
fdv_share = (sa_count["Persons"] / all_sa * 100).dropna()
fdv_share.name = "FDV-flagged % of all sexual assault"

mg.line_plot_finalise(
    fdv_share,
    title="FDV-Flagged Share of Police-Recorded Sexual Assault",
    ylabel="Per cent of all sexual assault victims",
    rfooter=SOURCE_POLICE,
    lfooter=LFOOTER + FDV_NOTE + "Calendar year. Rising share reflects changing police flagging practice.",
    width=2,
    color=["darkred"],
    marker="o",
    annotate=True,
    rounding=1,
    show=SHOW,
)

In [27]:
# pivot RCV 6 by sex (Number and Rate) for the all-sexual-assault series
all_sa_sex = rcv6[
    (rcv6["Characteristic"] == "Sex")
    & (rcv6["Subcategory"].isin(["Males", "Females"]))
].copy()
all_sa_sex["Value"] = pd.to_numeric(
    all_sa_sex["Value"].astype(str).str.replace(",", ""), errors="coerce"
)


def pivot_rcv6_sex(unit: str) -> pd.DataFrame:
    out = (
        all_sa_sex[all_sa_sex["Unit"] == unit]
        .pivot(index="Year", columns="Subcategory", values="Value")
        .sort_index()
    )
    out.index = pd.PeriodIndex(out.index.astype(int), freq="Y")
    return out[["Males", "Females"]]


all_sa_count = pivot_rcv6_sex("Number")
all_sa_rate = pivot_rcv6_sex("Victimisation rate (number per 100,000)")

# count chart: all police-recorded sexual assault by sex + OLS trend
all_sa_trend = pd.DataFrame({
    "Males (trend)": fit_trend(all_sa_count["Males"]),
    "Females (trend)": fit_trend(all_sa_count["Females"]),
})

ax = mg.line_plot(
    all_sa_count,
    color=["cornflowerblue", "hotpink"],
    width=2,
    annotate=True,
    rounding=0,
)
mg.line_plot(
    all_sa_trend,
    ax=ax,
    color=["cornflowerblue", "hotpink"],
    width=1,
    style="--",
    annotate=False,
)
mg.finalise_plot(
    ax,
    title="Police-Recorded Sexual Assault by Sex: Australia",
    ylabel="Police-recorded victims per year",
    rfooter=SOURCE_POLICE,
    lfooter=LFOOTER + "Calendar year. All police-recorded sexual assault (not FDV-restricted). Dashed lines = OLS linear trend.",
    legend={"loc": "upper left", "fontsize": "small"},
    show=SHOW,
)

In [28]:
# rate chart: all police-recorded sexual assault rate by sex + OLS trend
all_sa_rate_trend = pd.DataFrame({
    "Males (trend)": fit_trend(all_sa_rate["Males"]),
    "Females (trend)": fit_trend(all_sa_rate["Females"]),
})

ax = mg.line_plot(
    all_sa_rate,
    color=["cornflowerblue", "hotpink"],
    width=2,
    annotate=True,
    rounding=1,
)
mg.line_plot(
    all_sa_rate_trend,
    ax=ax,
    color=["cornflowerblue", "hotpink"],
    width=1,
    style="--",
    annotate=False,
)
mg.finalise_plot(
    ax,
    title="Police-Recorded Sexual Assault Rate by Sex: Australia",
    ylabel="Police-recorded victims per\n100,000 same-sex population",
    rfooter=SOURCE_POLICE,
    lfooter=LFOOTER + "Calendar year. All police-recorded sexual assault (not FDV-restricted). Crude rate. Dashed lines = OLS linear trend.",
    legend={"loc": "upper left", "fontsize": "small"},
    show=SHOW,
)

In [29]:
# line chart: FDV homicide victims by sex (police data) + linear trend (cross-check vs NHMP)
hom_main = hom_count[["Males", "Females"]]
hom_trend = pd.DataFrame({
    "Males (trend)": fit_trend(hom_count["Males"]),
    "Females (trend)": fit_trend(hom_count["Females"]),
})

ax = mg.line_plot(
    hom_main,
    color=["cornflowerblue", "hotpink"],
    width=2,
    annotate=True,
    rounding=0,
)
mg.line_plot(
    hom_trend,
    ax=ax,
    color=["cornflowerblue", "hotpink"],
    width=1,
    style="--",
    annotate=False,
)
mg.finalise_plot(
    ax,
    title="Police-Recorded FDV Homicide and Related Offences by Sex: Australia",
    ylabel="Police-recorded victims per year",
    rfooter=SOURCE_POLICE,
    lfooter=LFOOTER + FDV_NOTE + "Calendar year. Dashed lines = OLS linear trend.",
    legend={"loc": "upper right", "fontsize": "small"},
    show=SHOW,
)

## 4. Combined indexed view

Five Australia-level series indexed to 2019 = 100, with NHMD rebased to FY2020 (its first available year). Hard outcome measures — assault deaths, domestic homicide, FDV hospitalisations — track relatively flat. The two police-recorded series — all sexual assault and FDV-flagged sexual assault — rise sharply.

Police-recorded counts reflect victim reporting, police recording practices, and (for FDV-flagged) the share of recorded sexual assaults that get the FDV flag. All three have changed substantially over the period. The harder outcome measures are not subject to those filters. So the rising police-recorded lines tell us about reporting and flagging behaviour, not underlying incidence. On the available evidence, the claim that violence against women is escalating is not supported.

In [30]:
# indexed comparison: hard outcomes vs police-reported indicators
# Each series as a Persons rate per 100,000 population, indexed to 2019 = 100.
# NHMD rebased to FY2020 (its first available year).

BASE_YEAR = 2019

# Build Persons rate per 100k for series not already in rate form
hosp_rate_persons = fdv_hosp["Persons"] / mid_year_pop * 100_000
all_sa_rate_persons = all_sa / mid_year_pop * 100_000

raw_series = {
    "Assault deaths (rate)": assault_rate,
    "Domestic homicide (rate)": dom_rate["Persons"],
    "FDV hospitalisations (rate)": hosp_rate_persons,
    "All sexual assault, police-reported (rate)": all_sa_rate_persons,
    "FDV-flagged sexual assault, police-reported (rate)": sa_rate["Persons"],
}


def rebase(s: pd.Series, base_year: int) -> pd.Series:
    s = s.dropna()
    base_period = pd.Period(base_year, freq="Y")
    base_val = s.loc[base_period] if base_period in s.index else s.iloc[0]
    return s / base_val * 100


indexed = pd.DataFrame(
    {name: rebase(s, BASE_YEAR) for name, s in raw_series.items()}
)
indexed = indexed.loc[pd.Period("2010", "Y") : pd.Period("2025", "Y")]

mg.line_plot_finalise(
    indexed,
    title="Hard Outcomes vs Police-Reported Indicators: Indexed Comparison",
    ylabel="Index of rate per 100,000\n(2019 = 100)",
    rfooter="ABS 3303.0/4510.0; AIC NHMP; AIHW NHMD",
    lfooter=LFOOTER + "Persons total. NHMD rebased to FY2020.",
    width=2,
    color=["black", "darkred", "darkorange", "navy", "steelblue"],
    style=["-", "-", "-", "--", "--"],
    axhline={"y": 100, "color": "grey", "linestyle": ":"},
    annotate=False,
    legend={"loc": "upper left", "fontsize": "small"},
    show=SHOW,
)

## Finished

In [31]:
# watermark
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-04-26 11:51:13

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.13.0

conda environment: n/a

Compiler    : Clang 21.1.4 
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot : 0.2.23
numpy  : 2.4.4
pandas : 3.0.2
pathlib: 1.0.1
readabs: 0.1.8

Watermark: 2.6.0

